<a href="https://colab.research.google.com/github/sjsu-cs131-spring26/sec03-team9-tennis-sports/blob/colab/DRIVE_PA6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: This notebook is for actual testing. If data analysis tasks work succesfully here, please copy paste those code snippets to 'GCS_PA6.ipynb' notebook

In [ ]:
# feel free to run at start to ensure we are starting with clean cache
!rm -rf data.zip data/ *.parquet

### Environment setup + downloading data from drive

In [10]:
# environment setup

!pip install pyspark gdown -q
import os
import gdown
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, avg, lit, coalesce, count, broadcast, round, concat_ws, stddev


# initialize spark session
spark = (SparkSession.builder
  .appName("CS131_Sprint_5")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "8")
  .getOrCreate())

# download data from drive
file_id = "17UR9RRaoKasgV6UPWMoxwlJ7AUTkahTt"
url = f"https://drive.google.com/uc?id={file_id}"
output = "data.zip"

if not os.path.exists("data"):
  print("Fetching data...")
  gdown.download(url, output, quiet=True)
  !unzip -q -o data.zip
  print("Data loaded.")
else:
  print("Data already exists locally.")

Data already exists locally.


### Reading csv files into data frames: 'matches' 'players' 'rankings' and 'qual'



In [11]:
matches = (spark.read.option("header", True)
 .option("inferSchema", True)
 .csv("data/atp_matches.csv"))

players = (spark.read.option("header", True)
 .option("inferSchema", True)
 .csv("data/atp_players.csv"))

rankings = (spark.read.option("header", True)
 .option("inferSchema", True)
 .csv("data/atp_rankings.csv"))

qual = (spark.read.option("header", True)
 .option("inferSchema", True)
 .csv("data/atp_matches_qual_chall.csv"))

In [12]:
# A - cleaning

# initial checks on data tables
print("----Matches----")
matches.printSchema()
matches.show(3, truncate=False)
print(matches.count())


print("\n----Players----")
players.printSchema()
players.show(3, truncate=False)
print(players.count())

print("\n----Rankings----")
rankings.printSchema()
rankings.show(3, truncate=False)
print(rankings.count())

print("\n----Qual----")
qual.printSchema()
qual.show(3, truncate=False)
print(qual.count())

#clean

players_named = players.withColumn(
    "full_name",
    concat_ws(" ", col("name_first"), col("name_last"))
)

p_w = broadcast(players_named).alias("p_w")
p_l = broadcast(players_named).alias("p_l")

matches = (
    matches
    .select(
        "w_bpSaved",
        "w_bpFaced",
        "l_bpSaved",
        "l_bpFaced",
        "tourney_date",
        "tourney_level",
        "w_1stWon",
        "w_2ndWon",
        "w_svpt",
        "l_svpt",
        "winner_id",
        "loser_id",
        "surface",
        "winner_rank",
        "loser_rank",
        "w_1stIn",
        "l_1stIn",
        "l_ace",
        "w_ace",
        "w_df",
        "l_df",
        "l_2ndWon"
    )
    .filter(
      col("winner_id").isNotNull() &
      col("loser_id").isNotNull() &
      (col("w_svpt") > 0) &
      (col("l_svpt") > 0) &
      col("winner_rank").isNotNull() &
      col("loser_rank").isNotNull()
  )
)

m = matches.alias("m")

matches = (
    m
    .join(p_w, col("m.winner_id") == col("p_w.player_id"), "left")
    .join(p_l, col("m.loser_id") == col("p_l.player_id"), "left")
    .select(
        col("m.*"),
        col("p_w.full_name").alias("winner"),
        col("p_l.full_name").alias("loser")
    )
)

qual = (
    qual
    .select(
        "w_bpSaved",
        "w_bpFaced",
        "l_bpSaved",
        "l_bpFaced",
        "tourney_date",
        "tourney_level",
        "w_1stWon",
        "w_2ndWon",
        "w_svpt",
        "l_svpt",
        "winner_id",
        "loser_id",
        "surface",
        "winner_rank",
        "loser_rank",
        "w_1stIn",
        "l_1stIn",
        "l_ace",
        "w_ace",
        "w_df",
        "l_df",
        "l_2ndWon"
    )
    .filter(
        col("winner_id").isNotNull() &
        col("loser_id").isNotNull() &
        (col("w_svpt") > 0) &
        (col("l_svpt") > 0) &
        col("winner_rank").isNotNull() &
        col("loser_rank").isNotNull()
    )
)

q = qual.alias("q")

qual = (
    q
    .join(p_w, col("q.winner_id") == col("p_w.player_id"), "left")
    .join(p_l, col("q.loser_id") == col("p_l.player_id"), "left")
    .select(
        col("q.*"),
        col("p_w.full_name").alias("winner"),
        col("p_l.full_name").alias("loser")
    )
)


----Matches----
root
 |-- tourney_id: string (nullable = true)
 |-- tourney_name: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- draw_size: integer (nullable = true)
 |-- tourney_level: string (nullable = true)
 |-- tourney_date: integer (nullable = true)
 |-- match_num: integer (nullable = true)
 |-- winner_id: integer (nullable = true)
 |-- winner_seed: string (nullable = true)
 |-- winner_entry: string (nullable = true)
 |-- winner_name: string (nullable = true)
 |-- winner_hand: string (nullable = true)
 |-- winner_ht: integer (nullable = true)
 |-- winner_ioc: string (nullable = true)
 |-- winner_age: double (nullable = true)
 |-- loser_id: integer (nullable = true)
 |-- loser_seed: string (nullable = true)
 |-- loser_entry: string (nullable = true)
 |-- loser_name: string (nullable = true)
 |-- loser_hand: string (nullable = true)
 |-- loser_ht: integer (nullable = true)
 |-- loser_ioc: string (nullable = true)
 |-- loser_age: double (nullable = true)
 |-- s

### (B) Additional Analysis

In [13]:
# B - additional analysis tasks (Part 1)


# filter last 10 years, remove matches with 0 break points
# calculate winners bp_save_pct as bpSaved/bpFaced
recent_winners = (matches
  .filter(col("tourney_date") >= 20160101)
  .filter(col("w_bpFaced") > 0)
  .withColumn("bp_save_pct", col("w_bpSaved") / col("w_bpFaced")))

recent_winners_qual = (qual
  .filter(col("tourney_date") >= 20160101)
  .filter(col("w_bpFaced") > 0)
  .withColumn("bp_save_pct", col("w_bpSaved") / col("w_bpFaced")))

# calculate avg bp save % by tourney level
summary = (recent_winners
  .groupBy("tourney_level")
  .agg(round(avg(col("bp_save_pct")) * 100, 2).alias("winner_avg_bp_saved"))
  .orderBy("winner_avg_bp_saved", ascending=False))

summary_qual = (recent_winners_qual
  .groupBy("tourney_level")
  .agg(round(avg("bp_save_pct") * 100, 2).alias("winner_avg_bp_saved"))
  .orderBy("winner_avg_bp_saved", ascending=False))



print("Average break point saved % of winners in main draw (2016-2026):")
summary.show()

print("Average break point saved % of winners in qual draw (2016-2026):")
summary_qual.show()


Average break point saved % of winners in main draw (2016-2026):
+-------------+-------------------+
|tourney_level|winner_avg_bp_saved|
+-------------+-------------------+
|            F|              68.81|
|            G|               66.8|
|            A|              66.74|
|            D|              65.96|
|            M|              65.81|
+-------------+-------------------+

Average break point saved % of winners in qual draw (2016-2026):
+-------------+-------------------+
|tourney_level|winner_avg_bp_saved|
+-------------+-------------------+
|            A|              66.03|
|            M|              65.01|
|            G|              64.41|
|            C|              64.29|
+-------------+-------------------+



In [14]:
# B - additional analysis tasks (Part 2)

# main draw

# filtering last 10 years and svpt > 0
# calculating serving dominance metric
recent_matches = (matches
  .filter("tourney_date >= 20160101 AND w_svpt > 0")
  .withColumn("dom", (col("w_1stWon") + col("w_2ndWon")) / col("w_svpt")))

# average the serving dominance by surface
surface_summary = (recent_matches
  .groupBy("surface")
  .agg(round(avg("dom") * 100, 2).alias("avgSD"))
  .orderBy("avgSD", ascending=False))

# calculate volatility value
# group by winner and surface to get each player's average serving dominance per surface
# group by winner to find the standard deviation of those averages (v)
# filter out nulls (players who only played on one surface)
# average all individual volatility scores to get a single threshold
volatility = (recent_matches
  .groupBy("winner_id", "surface").agg(avg("dom").alias("p_avg"))
  .groupBy("winner_id").agg(stddev("p_avg").alias("v"))
  .filter(col("v").isNotNull())
  .agg(round(avg("v") * 100, 2).alias("avg_volatility_threshold")))

# qual draw

recent_matches_qual = (qual
  .filter("tourney_date >= 20160101 AND w_svpt > 0")
  .withColumn("dom", (col("w_1stWon") + col("w_2ndWon")) / col("w_svpt")))

surface_summary_qual = (recent_matches_qual
  .groupBy("surface")
  .agg(round(avg("dom") * 100, 2).alias("avgSD"))
  .orderBy("avgSD", ascending=False))

volatility_qual = (recent_matches_qual
  .groupBy("winner_id", "surface").agg(avg("dom").alias("p_avg"))
  .groupBy("winner_id").agg(stddev("p_avg").alias("v"))
  .filter(col("v").isNotNull())
  .agg(round(avg("v") * 100, 2).alias("avg_volatility_threshold")))

print("Surface Dominance (Main Draw):")
surface_summary.show()
print("Volatility (Main Draw):")
volatility.show()


print("Surface Dominance (Qual):")
surface_summary_qual.show()
print("Volatility (Qual):")
volatility_qual.show()

Surface Dominance (Main Draw):
+-------+-----+
|surface|avgSD|
+-------+-----+
|  Grass|71.15|
| Carpet|70.31|
|   Hard| 69.7|
|   Clay|67.53|
+-------+-----+

Volatility (Main Draw):
+------------------------+
|avg_volatility_threshold|
+------------------------+
|                     2.9|
+------------------------+

Surface Dominance (Qual):
+-------+-----+
|surface|avgSD|
+-------+-----+
| Carpet|74.12|
|  Grass|70.47|
|   Hard|68.62|
|   Clay|65.88|
+-------+-----+

Volatility (Qual):
+------------------------+
|avg_volatility_threshold|
+------------------------+
|                    2.98|
+------------------------+



In [15]:
# B - additional analysis tasks (Part 3)

# filter by last 10 years, loser rank exceeded winner rank by 25
upset_matches = (matches
  .filter(col("tourney_date") >= 20160101)
  .filter(col("winner_rank") > col("loser_rank") + 20)
  .filter((col("w_bpFaced") > 0) & (col("l_bpFaced") > 0))
  .filter((col("w_svpt") > col("w_1stIn")) & (col("l_svpt") > col("l_1stIn")))
)

    # calculating performance gap of winner vs loser
upset_stats = (upset_matches
  .withColumn("ace_gap", col("w_ace") - col("l_ace"))
  .withColumn("bp_save_gap", (col("w_bpSaved") / col("w_bpFaced")) - (col("l_bpSaved") / col("l_bpFaced")))
  .withColumn("first_serve_gap", (col("w_1stIn") / col("w_svpt")) - (col("l_1stIn") / col("l_svpt")))
  .withColumn("second_serve_win_gap", (col("w_2ndWon") / (col("w_svpt") - col("w_1stIn"))) - (col("l_2ndWon") / (col("l_svpt") - col("l_1stIn"))))
  .withColumn("df_gap", col("w_df") - col("l_df"))
)

underdog_report = upset_stats.agg(
  round(avg("ace_gap"), 2).alias("avg_extra_aces"),
  round(avg("bp_save_gap") * 100, 2).alias("avg_extra_bp_save_pct"),
  round(avg("first_serve_gap") * 100, 2).alias("avg_extra_1st_serve_pct"),
  round(avg("second_serve_win_gap") * 100, 2).alias("avg_extra_2nd_serve_win_pct"),
  round(avg("df_gap"), 2).alias("avg_df_diff")
)

underdog_report.show()

+--------------+---------------------+-----------------------+---------------------------+-----------+
|avg_extra_aces|avg_extra_bp_save_pct|avg_extra_1st_serve_pct|avg_extra_2nd_serve_win_pct|avg_df_diff|
+--------------+---------------------+-----------------------+---------------------------+-----------+
|          0.48|                 14.7|                   1.87|                        8.4|      -0.84|
+--------------+---------------------+-----------------------+---------------------------+-----------+

